# Libraries

In [34]:
import pandas as pd
import numpy as np
import os
import json
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from scipy.stats import spearmanr


In [30]:
# Directories
ROOT_DIR = os.path.abspath("")
DATA_DIR = os.path.join(ROOT_DIR, "data/")
print(DATA_DIR)

/Users/mba/Desktop/Cours/Machine_Learning/explaining-electricity-prices/data/


# Contexte

Les prix de marché de l'électricité dépendent de facteurs multiples. La première chose à avoir en tête est qu'il existe une multitude de marchés (à terme, day-ahead, intraday, balancing, ancillaires, etc). Dans ce projet, nous nous intéressons aux prix à terme de maturité 24h. Ces produits sont donc la livraison d'1MW d'électricité pour une durée de 24h.

Le but ici n'est pas de prédire le prix du jour, mais plutôt d'expliquer la variation journalière (prix observé pour le produit en J-1 vs J).

À présent, passons à la formation de ces prix. Il est important de comprendre que les prix à terme ne sont que le reflet de la moyenne pondérée de l'espérance des prix spots. Or ces derniers changent toutes les heures en fonction de l'équilibre offre/demande (on notera que la fréquence des prix spots est passée à 15 min depuis le 1er octobre 2025). Cet équilibre offre/demande permet la constitution d'un prix, lieu de rencontre des consommateurs d'un côté et des producteurs de l'autre. La demande étant relativement inélastique au prix, les consommateurs sont très souvent "price taker". Le prix pour une heure donnée reflête directement le coût marginal subit par la dernière centrale électrique appelé pour équilibrer le réseau (principe de "merit order").

Depuis quelques années, les marchés de l'électricité ont été impactés par de nombreux événements. Premièrement en 2022, après l'invasion de l'Ukraine par la Russie, avec de fortes hausses des prix du gaz, passant d'en moyenne 20€/MWh (pour le TTF, indice de prix de référence en EUROPE) à plus de 350€/MWh au pic de la crise. Comme expliqué précédemment, le prix de l'électricité est fixé par la dernière unité de production appelé. Losque cette production marginale provient d'une centrale à gaz, les prix spots s'envolent. Sachant qu'en général une centrale à gaz à un rendement entre 35% (cycle simple) et 60% (cycle combiné), il faut donc en moyenne 2 MWh de gaz pour produire 1MWh d'électricité. De tes coûts, sans compter les crédits carbone à sourcer pour couvrir les émissions engendrées par la centrale à gaz, ont suffit pour que les prix spots dépassent les 1000€/MWh sur certaines heures.

On comprend donc un premier problème. Les prix de l'électricité dépendent de la dernière unité appelée produite pour équilibrer le réseau.

Depuis 5 ans, on a aussi assisté à un développement exponentiel des capacités de production renouvelables (nous parlons ici du solaire et de l'éolien, l'hydraulique étant, surtout en France, déjà a son plein potentiel d'exploitation). Ce développement massif d'ENR a un impact sur les prix surtout en milieu de journée (moment où le soleil est à son zénith et où la production est maximale). Cette production massive d'électricité solaire augmente drastiquement l'offre électrique à un coût marginal quasiment nul. De plus, sur ces heures de milieu de journée la demande électrique est très souvent la plus basse de la journée. En effet, les besoins en lumière et en chauffage sont au plus bas. Ces deux phénomènes combinés entraînent des prix largement négatifs, surtout en période de "non stress" électrique, c'est-à-dire au printemps et en été.

On voit l'émergence d'une seconde problématique qui est le non alignement entre production abondante d'ENR et demande électrique.

Si on se replace dans le contexte de la crise de 2022, un autre problème mis en exergue est celui de la disponibilité du parc nucléaire français. Dans les années 1970, un grand plan de construction de centrales nucléaires a été développé afin de contrecarrer les impacts liés aux variations de prix du pétrole (le fameux, "En France, on a pas de pétrole mais on a des idées"). Ce qui faisait l'une des fierté de la France a été l'un des facteurs aggravant de la crise de 2022. En effet, durant cette crise la disponibilité du parc nucléaire français a été à son plus bas historique (les opérations de maintenance ainsi que les alertes sur certaines centrale se multipliant). De ce fait, on a du importer plus d'électricité depuis nos voisins et faire tourner des centrales à gaz qui ne tournaient pas avant. 

On aperçoit donc ici comme troisième problème (qui va de paire avec le second) le mix électrique du pays, et sa capacité à équilibrer son réseau par les échanges transfontaliers. Ici nous ne nous concentrerons que sur l'Allemagne et la France qui sont des pays assez largement connecté à leurs voisins au niveau électrique. Ce problème est donc moins important que pour des pays comme l'Espagne (cf le blackout d'avril 2025). 

Un dernier point à aborder pour finir cette mise en contexte est la différence qui existe entre le mix électrique français et le mix électrique allemand. En France, comme expliqué un peu plus haut, le mix électrique repose principalement sur du nucléaire (65-70%), des ENR (25-30%), et énergies fossiles qui sont uniquement sollicitées lors de forts pics de demandes. Cela nous permet d'avoir une production dite de "base" prévisible et résiliente mais aussi (petite fierté nationale) assez pilotable, en diminuant la production nucléaire sur les heures de production d'ENR afin de ne pas surcharger le réseau. En Allemagne, la politique énergétique menée depuis 10 ans a été basée sur une suppression des capacités nucléaires (0 centrale en 2023), ainsi qu'un développement massif d'ENR (mix composé en 2025 d'environ 50% d'ENR, 35% de charbon et 15% de gaz). Ce plan, qui paraît plus respectueux de l'environnement cache en fait un gros problème : l'intermittence. En période de faible ensoleillement ainsi que de manque de vent, le réseau est très vite saturé, et les besoins électriques sont colossaux. C'est à ce moment là que les prix grimpent avec des importations qui explosent.

Après ce bref tour d'horizon, on comprend bien que les marchés de l'électricité sont complexes et que ce qui forme le prix dépend d'énormément de facteurs spécifiques à chacun des pays. De ce fait, utilisé un seul modèle pour chaque pays serait omettre les différences intrinsèque qui existent entre chacun d'eux. 

# Data Management

## Data Import

In [32]:
# Importation des données
X_train = pd.read_csv(DATA_DIR + "X_train.csv")
y_train = pd.read_csv(DATA_DIR + "y_train.csv")
X_test = pd.read_csv(DATA_DIR + "X_test.csv")
y_test = pd.read_csv(DATA_DIR + "y_test_example.csv")

In [40]:
# On récupère les colonnes de X_train classées
with open(DATA_DIR + "facteurs.json") as f:
    columns_dict = json.load(f)

## QRT Benchmark check

In [21]:
lr = LinearRegression()

# Test set
X_train_clean = X_train.drop(['COUNTRY'], axis=1).fillna(0)
Y_train_clean = y_train['TARGET']

lr.fit(X_train_clean, Y_train_clean)

output_train = lr.predict(X_train_clean)

print(
    "Corrélation (Spearman) pour les données d'entraînement : {:.1f}%".format(
        100 * spearmanr(Y_train_clean, output_train).correlation
    )
)

# Train set
X_test_clean = X_test.drop(['COUNTRY'], axis=1).fillna(0)
Y_test_clean = y_test['TARGET']

output_test = lr.predict(X_test_clean)

print(
    "Corrélation (Spearman) pour les données de test : {:.1f}%".format(
        100 * spearmanr(Y_test_clean, output_test).correlation
    )
)

# Submission
Y_test_submission = X_test_clean[['ID']].copy()
Y_test_submission['TARGET'] = output_test
#Y_test_submission.to_csv('benchmark_qrt.csv', index=False)

Corrélation (Spearman) pour les données d'entraînement : 27.9%
Corrélation (Spearman) pour les données de test : -0.6%


## Data cleaning

In [49]:
# Affichage du nombre de valeurs manquantes par colonne
x = X_train.copy()
print(f"Train set observations: {len(x)}\n")
for key in columns_dict.keys():
    print(f"{key}:\n{x[columns_dict[key]].isna().sum()}\n")

print(f"Train set observations (without NaN): {len(x.dropna(how='any'))}")

Train set observations: 1494

to_keep:
ID         0
DAY_ID     0
COUNTRY    0
dtype: int64

production:
DE_GAS        0
FR_GAS        0
DE_COAL       0
FR_COAL       0
DE_HYDRO      0
FR_HYDRO      0
DE_NUCLEAR    0
FR_NUCLEAR    0
DE_SOLAR      0
FR_SOLAR      0
DE_WINDPOW    0
FR_WINDPOW    0
DE_LIGNITE    0
dtype: int64

consumption:
DE_CONSUMPTION      0
FR_CONSUMPTION      0
DE_RESIDUAL_LOAD    0
FR_RESIDUAL_LOAD    0
dtype: int64

exchange:
DE_FR_EXCHANGE     25
FR_DE_EXCHANGE     25
DE_NET_EXPORT     124
FR_NET_EXPORT      70
DE_NET_IMPORT     124
FR_NET_IMPORT      70
dtype: int64

weather:
DE_RAIN    94
FR_RAIN    94
DE_WIND    94
FR_WIND    94
DE_TEMP    94
FR_TEMP    94
dtype: int64

commodity_price:
GAS_RET       0
COAL_RET      0
CARBON_RET    0
dtype: int64

Train set observations (without NaN): 1276


In [ ]:
# Statistiques descriptives
pd.DataFrame(X_train).describe()

,ID,DAY_ID,DE_CONSUMPTION,FR_CONSUMPTION,DE_FR_EXCHANGE,FR_DE_EXCHANGE,DE_NET_EXPORT,FR_NET_EXPORT,DE_NET_IMPORT,FR_NET_IMPORT,...,FR_RESIDUAL_LOAD,DE_RAIN,FR_RAIN,DE_WIND,FR_WIND,DE_TEMP,FR_TEMP,GAS_RET,COAL_RET,CARBON_RET
count,1494.000000,1494.000000,1494.000000,1494.000000,1469.000000,1469.000000,1370.000000,1424.000000,1370.000000,1424.000000,...,1494.000000,1400.000000,1400.000000,1400.000000,1400.000000,1400.000000,1400.000000,1494.000000,1494.000000,1494.000000
mean,1072.759036,591.861446,0.427442,-0.020032,-0.145508,0.145508,-0.256332,-0.072643,0.256332,0.072643,...,-0.153688,-0.037831,0.019357,0.109480,0.123099,0.009451,0.008404,0.058126,0.061724,0.080510
std,618.013179,345.065043,0.673412,0.918995,0.970226,0.970226,0.957443,1.075830,0.957443,1.075830,...,0.896325,0.984233,1.051781,1.056243,1.054692,0.972394,1.003356,1.097768,1.033853,1.098624
min,0.000000,0.000000,-2.265563,-1.462350,-2.856874,-2.634831,-2.464849,-2.825331,-2.279619,-1.951516,...,-1.678936,-2.128531,-1.726420,-1.880419,-1.895319,-4.549638,-5.787097,-5.349463,-5.706442,-4.281790
25%,540.250000,292.250000,-0.037421,-0.716771,-0.875213,-0.638867,-0.977214,-0.851500,-0.452252,-0.794843,...,-0.802333,-0.642117,-0.503927,-0.652135,-0.672614,-0.618259,-0.647948,-0.624238,-0.458038,-0.522968
50%,1077.500000,591.000000,0.357061,-0.394166,-0.164287,0.164287,-0.306899,0.099455,0.306899,-0.099455,...,-0.460160,-0.274901,-0.228147,-0.261571,-0.229031,-0.026306,-0.020889,0.008493,0.063312,0.054056
75%,1597.500000,885.750000,0.922057,0.650533,0.638867,0.875213,0.452252,0.794843,0.977214,0.851500,...,0.382191,0.335237,0.154351,0.635050,0.824781,0.651832,0.699131,0.676415,0.641446,0.599094
max,2146.000000,1215.000000,2.033851,3.300640,2.634831,2.856874,2.279619,1.951516,2.464849,2.825331,...,2.918326,7.756118,9.473201,5.085624,4.965028,2.858758,2.817239,5.674778,3.746576,5.471818


In [78]:
# Détection des outliers
def get_outliers(serie):
    Q1 = serie.quantile(0.25)
    Q3 = serie.quantile(0.75)
    IQR = Q3 - Q1

    borne_inf = Q1 - 1.5 * IQR
    borne_sup = Q3 + 1.5 * IQR

    outliers = serie[(serie < borne_inf) | (serie > borne_sup)]
    if not len(outliers)==0:
        print(f"{serie.name}:\nNombre d'outliers : {len(outliers)}")
        #print(f"Valeurs aberrantes : {outliers.values}")

for col in X_train.drop(columns=['ID', 'DAY_ID', 'COUNTRY']).columns:
    get_outliers(X_train[col])


DE_CONSUMPTION:
Nombre d'outliers : 9
FR_CONSUMPTION:
Nombre d'outliers : 2
FR_COAL:
Nombre d'outliers : 113
DE_HYDRO:
Nombre d'outliers : 10
FR_HYDRO:
Nombre d'outliers : 28
DE_NUCLEAR:
Nombre d'outliers : 28
DE_WINDPOW:
Nombre d'outliers : 62
FR_WINDPOW:
Nombre d'outliers : 38
DE_LIGNITE:
Nombre d'outliers : 30
DE_RESIDUAL_LOAD:
Nombre d'outliers : 28
FR_RESIDUAL_LOAD:
Nombre d'outliers : 21
DE_RAIN:
Nombre d'outliers : 90
FR_RAIN:
Nombre d'outliers : 142
DE_WIND:
Nombre d'outliers : 52
FR_WIND:
Nombre d'outliers : 14
DE_TEMP:
Nombre d'outliers : 15
FR_TEMP:
Nombre d'outliers : 10
GAS_RET:
Nombre d'outliers : 36
COAL_RET:
Nombre d'outliers : 66
CARBON_RET:
Nombre d'outliers : 64


In [59]:
# On ne garde que les lignes sans valeurs manquantes
X_train_clean = (
    X_train
    .dropna(how="any")
    .sort_values(by="ID")
    .reset_index(drop=True)
)
y_train_clean = y_train[y_train["ID"].isin(X_train_clean["ID"])].sort_values(by="ID").reset_index(drop=True)

## Features creation

Features Possibles : 
- (Residual Load + NET IMPORT + NET EXPORT) FR => Notion d'elec manquante pour équilibrer le réseau 
- (Residual Load + NET IMPORT + NET EXPORT) DE => Notion de manque d'élec chez le voisin
- Marginal cost gas and coal => Notion de coût marginal 
=> Gas efficiency * Gas price + Gas emission factor * Carbon Ret       
=> Coal efficiency * Coal price + Coal emission factor * Carbon Ret
- TEMP
- RAIN
- WIND 

In [27]:
# Ajout des features sur les deux datasets
for X in [X_train_clean, X_test_clean]:

    # Déséquilibre offre / demande
    # Allemagne
    X["DE_BALANCE"] = (
        X["DE_GAS"] + X["DE_COAL"] + X["DE_LIGNITE"] +
        X["DE_NUCLEAR"] + X["DE_SOLAR"] + X["DE_WINDPOW"] + X["DE_HYDRO"]
    ) - X["DE_CONSUMPTION"]

    # France
    X["FR_BALANCE"] = (
        X["FR_GAS"] + X["FR_COAL"] +
        X["FR_NUCLEAR"] + X["FR_SOLAR"] + X["FR_WINDPOW"] + X["FR_HYDRO"]
    ) - X["FR_CONSUMPTION"]

    # Poids des ENR
    X["DE_RENEWABLE_SHARE"] = (X["DE_SOLAR"] + X["DE_WINDPOW"]) / (X["DE_CONSUMPTION"] + 1e-6)
    X["FR_RENEWABLE_SHARE"] = (X["FR_SOLAR"] + X["FR_WINDPOW"]) / (X["FR_CONSUMPTION"] + 1e-6)

    # Dépendance au gaz
    X["DE_GAS_SHARE"] = X["DE_GAS"] / (X["DE_CONSUMPTION"] + 1e-6)
    X["FR_GAS_SHARE"] = X["FR_GAS"] / (X["FR_CONSUMPTION"] + 1e-6)

    # Interaction avec commodités
    X["DE_GAS_IMPACT"] = X["DE_GAS"] * X["GAS_RET"]
    X["FR_GAS_IMPACT"] = X["FR_GAS"] * X["GAS_RET"]

    X["DE_CARBON_IMPACT"] = X["DE_COAL"] * X["CARBON_RET"]
    X["FR_CARBON_IMPACT"] = X["FR_GAS"] * X["CARBON_RET"]

    # Flux transfrontaliers
    X["NET_FLOW"] = X["FR_DE_EXCHANGE"] - X["DE_FR_EXCHANGE"]
    X["FLOW_IMBALANCE"] = X["FR_NET_EXPORT"] - X["DE_NET_EXPORT"]

    # Variabilité ENR
    X["DE_WIND_SOLAR"] = X["DE_WINDPOW"] + X["DE_SOLAR"]
    X["FR_WIND_SOLAR"] = X["FR_WINDPOW"] + X["FR_SOLAR"]

    # Différences entre les pays, capte les arbitrages marché
    X["NUCLEAR_DIFF"] = X["FR_NUCLEAR"] - X["DE_NUCLEAR"]
    X["COAL_DIFF"] = X["DE_COAL"] - X["FR_COAL"]
    X["RENEWABLE_DIFF"] = (
        (X["DE_SOLAR"] + X["DE_WINDPOW"]) -
        (X["FR_SOLAR"] + X["FR_WINDPOW"])
    )

## Standardisation

In [28]:
cols_to_scale = [
    col for col in X_train_clean.columns 
    if col not in columns_dict["to_keep"]
]

scaler = StandardScaler()

X_train_scaled = X_train_clean.copy()
X_train_scaled[cols_to_scale] = scaler.fit_transform(X_train_clean[cols_to_scale])

X_test_scaled = X_test_clean.copy()
X_test_scaled[cols_to_scale] = scaler.transform(X_test_clean[cols_to_scale])


## ALL/FR/DE datasets creation 

In [30]:
# Colonnes FR/DE
fr_cols = list(dict.fromkeys(columns_dict["to_keep"] + [col for col in X_train_scaled.columns if "FR" in col]))
de_cols = list(dict.fromkeys(columns_dict["to_keep"] + [col for col in X_train_scaled.columns if "DE" in col]))

# Séparation train
X_train_FR = (
    X_train_scaled[fr_cols]
    .query("COUNTRY == 'FR'")
    .reset_index(drop=True)
)
y_train_FR = (
    y_train_clean[y_train_clean["ID"].isin(X_train_FR["ID"])]
    .reset_index(drop=True)
)

X_train_DE = (
    X_train_scaled[de_cols]
    .query("COUNTRY == 'DE'")
    .reset_index(drop=True)
)
y_train_DE = (
    y_train_clean[y_train_clean["ID"].isin(X_train_DE["ID"])]
    .reset_index(drop=True)
)

# Séparation test
X_test_FR = (
    X_test_scaled[fr_cols]
    .query("COUNTRY == 'FR'")
    .reset_index(drop=True)
)
y_test_FR = (
    y_test_clean[y_test_clean["ID"].isin(X_test_FR["ID"])]
    .reset_index(drop=True)
)

X_test_DE = (
    X_test_scaled[de_cols]
    .query("COUNTRY == 'DE'")
    .reset_index(drop=True)
)
y_test_DE = (
    y_test_clean[y_test_clean["ID"].isin(X_test_DE["ID"])]
    .reset_index(drop=True)
)


In [ ]:
# Jeux finaux
train_dict = {
    "ALL": pd.merge(X_train_scaled, y_train_clean, on="ID"),
    "FR": pd.merge(X_train_FR, y_train_FR, on="ID"),
    "DE": pd.merge(X_train_DE, y_train_DE, on="ID")
}

test_dict = {
    "ALL": pd.merge(X_test_scaled, y_test_clean, on="ID"),
    "FR": pd.merge(X_test_FR, y_test_FR, on="ID"),
    "DE": pd.merge(X_test_DE, y_test_DE, on="ID")
}


In [26]:
X_train_FR

,ID,DAY_ID,COUNTRY,FR_CONSUMPTION,DE_FR_EXCHANGE,FR_DE_EXCHANGE,FR_NET_EXPORT,FR_NET_IMPORT,FR_GAS,FR_COAL,...,FR_RESIDUAL_LOAD,FR_RAIN,FR_WIND,FR_TEMP,FR_BALANCE,FR_RENEWABLE_SHARE,FR_GAS_SHARE,FR_GAS_IMPACT,FR_CARBON_IMPACT,FR_WIND_SOLAR
0,1179,1,FR,1.549737,0.481228,-0.481228,0.802058,-0.802058,1.820403,-0.633412,...,1.759024,-0.474061,-1.555613,0.221013,1.351376,-0.017739,0.041718,2.671507,1.694278,-0.658070
1,1327,2,FR,-0.641649,-0.984531,0.984531,0.327737,-0.327737,0.087368,-0.590756,...,-0.337533,-0.355408,-1.051156,0.631308,-0.870645,-0.020210,0.031041,0.764820,0.418752,-0.761876
2,2016,3,FR,-0.835531,-0.933096,0.933096,-0.461934,0.461934,-0.352476,-0.485935,...,-0.714664,-1.061608,0.271557,-0.963556,0.013003,-0.031383,0.033824,0.049675,-0.013639,0.954691
3,2047,5,FR,-0.413155,0.290267,-0.290267,-1.551890,1.551890,0.166484,-0.588926,...,-0.352151,-0.750952,-0.164870,1.451140,0.199018,-0.046082,0.029029,-0.033760,0.419138,1.381840
4,1995,7,FR,-0.593212,0.144674,-0.144674,-0.582855,0.582855,0.391536,-0.614977,...,-0.655000,-0.631080,1.093180,0.944962,-0.263445,-0.035024,0.028847,-0.226152,-0.037529,0.896006
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
675,1947,1201,FR,0.826584,0.905316,-0.905316,0.689872,-0.689872,-0.329318,-0.601689,...,-0.060180,0.217407,3.060269,1.630284,1.039023,0.012906,0.034899,0.051565,-0.029251,2.510235
676,1498,1202,FR,-1.328748,-1.085758,1.085758,0.657436,-0.657436,-1.754566,-0.607544,...,-1.316044,-0.947185,-0.912458,1.402593,-0.391519,-0.023846,0.038492,-0.172174,-0.629896,0.200188
677,1120,1205,FR,0.016113,0.532619,-0.532619,0.190937,-0.190937,-0.275418,2.336319,...,0.284690,-0.415435,-0.809029,-0.493689,0.016426,-0.006915,0.027885,0.028116,-0.147971,-1.133540
678,2039,1208,FR,-0.689919,0.580031,-0.580031,-1.430738,1.430738,-0.227613,-0.572642,...,-0.487605,2.515516,0.403912,-1.452393,-1.434032,-0.022755,0.033038,0.069387,0.062843,-0.427490


# First Models

In [24]:
models = {
    "Linear": LinearRegression(),
    "Ridge (alpha = 1)": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.1, max_iter=10000)
}

cols_to_drop = columns_dict["to_keep"]  # Colonnes à ne pas utiliser pour l'entraînement

results = []

for dataset_name in train_dict:
    data_train = train_dict[dataset_name]
    data_test = test_dict[dataset_name]

    y_train = data_train["TARGET"]
    X_train = data_train.drop(columns=["TARGET"] + cols_to_drop, errors="ignore")

    y_test = data_test["TARGET"]
    X_test = data_test.drop(columns=["TARGET"] + cols_to_drop, errors="ignore")

    for model_name, model in models.items():
        model.fit(X_train, y_train)

        y_pred_train = model.predict(X_train)
        y_pred_test = model.predict(X_test)

        results.append({
            "Dataset": dataset_name,
            "Model": model_name,
            "Spearman Train": spearmanr(y_train, y_pred_train).correlation,
            "Spearman Test": spearmanr(y_test, y_pred_test).correlation
        })

results_df = (
    pd.DataFrame(results)
    .sort_values(by="Spearman Test", ascending=False)
    .reset_index(drop=True)
)

results_df

,Dataset,Model,Spearman Train,Spearman Test
0,DE,Lasso,0.363445,0.024976
1,DE,Ridge (alpha = 1),0.413949,0.011519
2,DE,Linear,0.413600,0.011361
3,ALL,Lasso,0.227023,-0.018582
4,FR,Ridge (alpha = 1),0.180067,-0.027294
5,FR,Linear,0.175732,-0.028710
6,ALL,Ridge (alpha = 1),0.296785,-0.030094
7,ALL,Linear,0.295849,-0.031133
8,FR,Lasso,0.132695,-0.045495


# Features Research